## Libraries & Function

In [1]:
import numpy as np
from scipy.linalg import expm
from qutip import *
import numba
from numba import njit, prange
import os
import time

In [2]:

sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) ; sp = np.array(([[0,1],[0,0]]), dtype=complex) ; sm = np.array(([[0,0],[1,0]]), dtype=complex)


In [3]:
#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

### Hamiltonians and U operator

In [4]:
def system_Hamiltonian(E):
    """
    Build up of the System's Hamiltonian for a single qubit.
    
    Parameters: - E: Float, Energy difference between excited and ground state
        
    Returns : System's Hamiltonian as Numpy array
    """
    # Using sz defined in Cell 2
    H_sys = (E / 2) * sz 
    
    return H_sys

In [5]:
def interaction_Hamiltonian(c_CM):   
    """
    Build up of the Hamiltonian of Interaction for the Collision System - Ancilla.
    Implements the energy exchange model: c * (sigma_minus @ sigma_plus^a + sigma_plus @ sigma_minus^a)
       
    Parameters: - c_CM : float, Interaction Force for the System-Ancilla collision
        
    Returns : Hamiltonian of Interaction as Qutip object
    """
    # Tensor product for relaxation: System relaxes (sm), Ancilla excites (sp)
    term_relaxation = tensor(Qobj(sm), Qobj(sp))
    
    # Tensor product for excitation: System excites (sp), Ancilla relaxes (sm)
    term_excitation = tensor(Qobj(sp), Qobj(sm))
    
    # Total Collisional Hamiltonian
    H_int = (c_CM * (term_relaxation + term_excitation)).full()
    
    return H_int

In [6]:
def complete_Hamiltonian(Delta_E, c_CM):
    """
    Generation of 3 Hamiltonians for the single qubit collision model:
                - H_system : system Hamiltonian
                - H_collision : interaction Hamiltonian with 1 ancilla
                - H_tot : complete Hamiltonian (system + collision)

    Parameters: - Delta_E: Float, System Energy difference
                - c_CM : float, Interaction Force
                
    Returns : H_system, H_collision, H_tot
    """
    # 1. System Hamiltonian (using the function defined previously)
    H_system = system_Hamiltonian(Delta_E)
    
    # 2. Collision Hamiltonian
    H_collision = interaction_Hamiltonian(c_CM) 
    
    # 3. Total Hamiltonian
    # Expand H_sys in the total space: System tensor Identity_ancilla
    Id_ancilla = qeye(2)
    H_system_expanded = tensor(Qobj(H_system), Id_ancilla)  
        
    H_tot = H_system_expanded + H_collision
        
    return H_system, H_collision, H_tot

In [7]:
def evolution_operator(H, dt, method='expm', hermitian=True):
    """
    Build up of the evolution operator U = exp(-i H dt) using Expm or analytic diagonalization.
   
    Parameters: - H : Qobj or nparray, System Hamiltonian
                - dt : float, Timestep
    
    Method : - "expm"-> build up of the Matrix Exponential with expm
             - "diagonalization"->  build up of the propagater U as V @(exp(-i W dt))@ V_dag with W eigenvalues and V eigenvector of the Hamiltonian 

    Returns : Evolution Operator U, 
    """
    H = H.full() if hasattr(H, "full") else np.array(H)
    
    # -----------
    # Expm method
    # -----------
    
    if method == 'expm':
        U = expm(-1j * H * dt)
        return U
        
    # ---------------
    # Diagonalization
    # ---------------
    
    elif method == 'diagonalization':
        if hermitian:
            w, V = np.linalg.eigh(H)
            V_inv = V.conj().T
        else:
            w, V = np.linalg.eig(H) 
            V_inv = np.linalg.inv(V)
                
        U_diag = np.diag(np.exp(-1j * w * dt))
        U = V @ U_diag @ V_inv
        return U, U_diag, w, V

    else:
        raise ValueError("method must be 'expm' or 'diagonalization'")


### Lindblad functions

In [8]:
def Liouvillian(H, gamma_k, L_k):
    """
    Build the Liouvillian superoperator using row-major convention (NumPy).
    
    Parameters: - H : nparray, Hamiltonian matrix
                - gamma_k : list, Decay rates
                - L_k : list, Jump Operators
    
    Returns: - super_L : nparray, Liouvillian superoperator
    """    
    I = np.eye(H.shape[0], dtype=complex)
    
    # Unitary evolution: -i * [H, rho]
    super_L = -1.j * (np.kron(H, I) - np.kron(I, H.T))
    
    # Dissipator terms
    for k in range(len(gamma_k)):
        L = L_k[k]
        L_dag = np.conj(L).T
        L_dag_L = L_dag @ L
        
        super_L += gamma_k[k] * (np.kron(L, np.conj(L)) - 0.5 * np.kron(L_dag_L, I) - 0.5 * np.kron(I, L_dag_L.T))
    
    return super_L


In [9]:
@njit(cache=True)
def _evolve_expm_core(super_U, rho_vec_initial, n_times):
    """
    Core evolution loop with expm method (Numba JIT)
    """
    rho_size = rho_vec_initial.shape[0]
    rho_vec_list = np.zeros((rho_size, n_times), dtype=np.complex128)
    rho_vec_list[:, 0] = rho_vec_initial
    
    for i in range(1, n_times):
        rho_vec_list[:, i] = super_U @ rho_vec_list[:, i - 1]
    
    return rho_vec_list


@njit(cache=True)
def _evolve_diagonal_core(V, V_inv, U_diag, rho_vec_initial, n_times):
    """
    Core evolution loop with diagonal method (Numba JIT)
    """
    n_states = len(U_diag)
    
    # Initial coefficients in eigenbasis
    coeff = V_inv @ rho_vec_initial
    coeff_list = np.zeros((n_states, n_times), dtype=np.complex128)
    coeff_list[:, 0] = coeff
    
    # Evolution of coefficients
    for i in range(1, n_times):
        coeff_list[:, i] = U_diag * coeff_list[:, i - 1]
    
    # Transform back to original basis
    rho_vec_list = V @ coeff_list
    
    return rho_vec_list


def Lindblad_evo(rho, H, gamma_k, L_k, times, method="expm", vectorized=True):
    """
    Evolution of the density matrix with the Lindblad Eq. (Optimized with Numba)
    
    Method: - "expm" -> propagator = expm(super_L * dt)
            - "diagonal" -> diagonalization of the super-operator
        
    Vectorized: True/False to choose the output format
    
    Parameters: - H : nparray, System Hamiltonian
                - rho : Qobj or nparray, Initial Density Matrix
                - gamma_k : list, List of Decay Rates
                - L_k : list, List of Jump Operators
                - times : array, Time array
        
    Returns : - if vectorized=True → array (N^2, Nt)
              - if vectorized=False → array (Nt, N_site, N_site)
              - if method="diagonal" also returns V, W
    """
    # Convert to NumPy
    L_k = [L.full() if hasattr(L, "full") else np.array(L, dtype=complex) for L in L_k]
    H = H.full() if hasattr(H, "full") else np.array(H, dtype=complex)
    rho = rho.full() if hasattr(rho, "full") else np.array(rho, dtype=complex)
    
    rho_shape = H.shape[0]
    dt = times[1] - times[0]
    n_times = len(times)
    
    # Build Liouvillian
    super_L = Liouvillian(H, gamma_k, L_k)
    
    # Vectorize initial state
    rho_vec = rho.reshape(rho_shape * rho_shape)
    
    # -------------
    # Expm method
    # -------------
    if method == "expm":
        # Compute propagator 
        super_U = expm(super_L * dt)
        
        # evolution loop
        rho_vec_list = _evolve_expm_core(super_U, rho_vec, n_times)
        
        # Output
        if vectorized:
            return rho_vec_list
        else:
            return rho_vec_list.T.reshape(n_times, rho_shape, rho_shape)
    
    # ------------------
    # Diagonal method
    # ------------------
    elif method == "diagonal":
        # Diagonalize Liouvillian 
        W, V = np.linalg.eig(super_L)
        V_inv = np.linalg.inv(V)
        
        # Diagonal evolution operator
        U_diag = np.exp(W * dt)
        
        # evolution loop
        rho_vec_list = _evolve_diagonal_core(V, V_inv, U_diag, rho_vec, n_times)
        
        # Output
        if vectorized:
            return rho_vec_list, V, W
        else:
            return rho_vec_list.T.reshape(n_times, rho_shape, rho_shape), V, W
    
    else:
        raise ValueError("method must be 'expm' or 'diagonal'")

### Isolated system

In [ ]:
@njit(cache=True)
def _compute_trajectory_isolated_core_single(psi_initial, U_site, projectors, projectors_cohe, n_times):
    """
    Core evolution loop optimized with Numba for a single qubit.
    Computes both populations (real) and coherences (complex) over time.
    """
    N_proj = projectors.shape[0]
    N_cohe = projectors_cohe.shape[0]
    
    # Pre-allocate arrays
    pop_traj = np.zeros((N_proj, n_times), dtype=np.float64)
    coh_traj = np.zeros((N_cohe, n_times), dtype=np.complex128)
    
    # Initial state values
    for p in range(N_proj):
        P_psi = projectors[p] @ psi_initial
        pop_traj[p, 0] = np.real(np.vdot(psi_initial, P_psi))
        
    for c in range(N_cohe):
        P_cohe_psi = projectors_cohe[c] @ psi_initial
        coh_traj[c, 0] = np.vdot(psi_initial, P_cohe_psi)
    
    # Time Evolution
    psi = psi_initial.copy()
    for step in range(1, n_times):
        psi = U_site @ psi
        
        # Store populations and coherences for the current step
        for p in range(N_proj):
            P_psi = projectors[p] @ psi
            pop_traj[p, step] = np.real(np.vdot(psi, P_psi))
            
        for c in range(N_cohe):
            P_cohe_psi = projectors_cohe[c] @ psi
            coh_traj[c, step] = np.vdot(psi, P_cohe_psi)

    return pop_traj, coh_traj

def compute_trajectory_wf_isolated(times, projectors, projectors_cohe, psi_sys_initial, U_site):
    """
    Optimized isolated system evolution with Numba for a single qubit.
    Calculates both populations and coherences.
    """
    # Convert to NumPy
    U_site_np = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_initial_np = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    
    # Flatten if needed
    if psi_initial_np.ndim > 1:
        psi_initial_np = psi_initial_np.flatten()

    # Time parameters
    n_times = len(times)
        
    # Call JIT-compiled function
    pop_traj_isolated, coh_traj_isolated = _compute_trajectory_isolated_core_single(
        psi_initial_np, U_site_np, projectors, projectors_cohe, n_times
    )

    return pop_traj_isolated, coh_traj_isolated

### Collisional Method functions

#### Evolution with $ U_{complete} $ and then trace on the ancilla

In [ ]:
@njit(cache=True)
def _compute_trace_ancilla_core_single(rho_sys, rho_anc, U_step, U_step_dag, projectors, projectors_cohe, n_times, dim_sys, dim_anc):
    """
    Core computation optimized with Numba for a single qubit + single ancilla.
    """
    N_state = projectors.shape[0]
    pops_complete = np.zeros((N_state, n_times), dtype=np.float64)
    cohe_omplete = np.zeros((N_state, n_times), dtype=np.complex128)
    
    # Initial state expectations for all projectors
    for p in range(N_state):
        pops_complete[p, 0] = np.real(np.trace(projectors[p] @ rho_sys))
        cohe_complete[p, 0] = np.trace(projectors_cohe[p] @ rho_sys)
    
    # Time Evolution
    for t in range(1, n_times):
        # 1: Expansion (System tensor Ancilla)
        rho_tot = np.kron(rho_sys, rho_anc)
        
        # 2: Evolution
        rho_tot = U_step @ rho_tot @ U_step_dag
        
        # 3: Partial Trace over the Ancilla
        rho_tot_reshaped = rho_tot.reshape(dim_sys, dim_anc, dim_sys, dim_anc)
        
        # Manual trace 
        rho_sys = np.zeros((dim_sys, dim_sys), dtype=np.complex128)
        for i in range(dim_sys):
            for j in range(dim_sys):
                for k in range(dim_anc):
                    rho_sys[i, j] += rho_tot_reshaped[i, k, j, k]
        
        # 4: Store populations for the current step
        for p in range(N_state):
            pops_complete[p, t] = np.real(np.trace(projectors[p] @ rho_sys))
            cohe_complete[p, t] = np.trace(projectors_cohe[p] @ rho_sys)
    
    return pops_complete, cohe_complete


def compute_trace_ancilla(rho_sys_initial, rho_anc_single, U_diag, V, times, projectors):
    """
    Evolution with complete collisional Hamiltonian and then trace on the Ancilla 
    degrees of freedom for a single qubit system. Corresponds to an average 
    over an infinite number of trajectories.
    """

    # For a single qubit system, the total ancilla is just the single ancilla state
    rho_anc = rho_anc_single.full() if hasattr(rho_anc_single, 'full') else rho_anc_single.copy()
    
    # Convert system state to numpy
    rho_sys = rho_sys_initial.full() if hasattr(rho_sys_initial, 'full') else rho_sys_initial.copy()

    # Time parameters
    n_times = len(times)

    # Dimensions
    dim_sys = rho_sys.shape[0]
    dim_anc = rho_anc.shape[0]
    
    # Evolution operator for a single time step
    V_np = V.full() if hasattr(V, 'full') else V
    U_diag_np = U_diag.full() if hasattr(U_diag, 'full') else U_diag
    U_step = V_np @ U_diag_np @ V_np.conj().T
    U_step_dag = U_step.conj().T
    
    # Call JIT-compiled function
    pops_complete, cohe_complete = _compute_trace_ancilla_core_single(
        rho_sys, rho_anc, U_step, U_step_dag, projectors, projectors_cohe, n_times, dim_sys, dim_anc
    )
    
    return pops_complete, cohe_complete

#### Single trajectory Quantum Jump

In [ ]:
@njit
def sigma_xyz_expectation_value(psi, sx, sy, sz):
    """
    Function to calculate expectation value of <sigmax>, <sigmay>, <sigmaz> 
    using matrix multiplication for a single qubit.

    Parameters: - psi : nparray, wave function at time t (shape: 2,)
                - sx : nparray, Pauli X operator matrix
                - sy : nparray, Pauli Y operator matrix
                - sz : nparray, Pauli Z operator matrix

    returns : - S_x, float expectation value of <sigmax>
              - S_y, float expectation value of <sigmay>
              - S_z, float expectation value of <sigmaz>
    """
    
    S_x = np.real(np.vdot(psi, sx @ psi))
    S_y = np.real(np.vdot(psi, sy @ psi))
    S_z = np.real(np.vdot(psi, sz @ psi))

    return S_x, S_y, S_z

In [ ]:
@njit
def compute_Bloch_Sphere(psi):
    """
    Function to extract the expectation value of the Bloch's Sphere components 
    <sigmax>, <sigmay>, <sigmaz> for a single qubit system.

    Parameters: - psi : nparray, wave function at time t (shape: 2,)

    Returns: - r_x_step, float expectation value of x component <sigmax>
             - r_y_step, float expectation value of y component <sigmay>
             - r_z_step, float expectation value of z component <sigmaz>
    """

    # Extract the probability amplitudes for |0> and |1>
    c_0 = psi[0] 
    c_1 = psi[1]
    
    # Complex conjugate of c_0
    c_0_conj = np.conj(c_0) 

    # Bloch vector components derived analytically
    # <sigma_x> = c_0^* c_1 + c_1^* c_0 = 2 * Re(c_0^* * c_1)
    r_x_step = 2 * np.real(c_0_conj * c_1)
    
    # <sigma_y> = -i(c_0^* c_1 - c_1^* c_0) = 2 * Im(c_0^* * c_1)
    r_y_step = 2 * np.imag(c_0_conj * c_1)

    # <sigma_z> = |c_0|^2 - |c_1|^2
    r_z_step = np.abs(c_0)**2 - np.abs(c_1)**2 

    return r_x_step, r_y_step, r_z_step

In [ ]:
def generate_kraus_operators(c_CM, dt, mode="QJ"):
    """
    Generate the Kraus operators M0 and M1 for the single qubit collision model.
    
    Parameters: - c_CM: float, Interaction coefficient
                - dt: float, Time step
                - mode: str, "QJ" for Quantum Jump, "SD" for State Diffusion
                
    Returns: - M0, M1: np.ndarray, The two Kraus operators
    """
    c_dt = c_CM * dt
    cos_val = np.cos(c_dt)
    sin_val = np.sin(c_dt)
    
    if mode == "QJ":
        # Quantum Jump: Measurement in sigma_z basis
        M0 = np.array([[1.0, 0.0], 
                       [0.0, cos_val]], dtype=np.complex128)
                       
        M1 = np.array([[0.0, -1j * sin_val], 
                       [0.0, 0.0]], dtype=np.complex128)
                       
    elif mode == "SD":
        # State Diffusion: Measurement in sigma_x basis (superposition)
        M0 = (1 / np.sqrt(2)) * np.array([[1.0, -1j * sin_val], 
                                          [0.0, cos_val]], dtype=np.complex128)
                                          
        M1 = (1 / np.sqrt(2)) * np.array([[1.0, 1j * sin_val], 
                                          [0.0, cos_val]], dtype=np.complex128)
    else:
        raise ValueError("Mode must be 'QJ' or 'SD'")
        
    return M0, M1

In [ ]:
@njit(parallel=True, cache=True, fastmath=True)
def compute_trajectory_wf_core_single(psi_initial, U_site, M0, M1, 
                                      projectors, projectors_cohe,
                                      N_traj, n_times, seeds):
    """
    Core trajectory evolution optimized with Numba for a single qubit.
    Probabilities are dynamically computed at each time step.
    """
    N_state = projectors.shape[0]
    
    # Pre-allocate arrays for N_traj
    pop_traj = np.zeros((N_state, n_times, N_traj), dtype=np.float64)
    coh_traj = np.zeros((N_state, n_times, N_traj), dtype=np.complex128)
    
    # Initialization at t=0
    for i in range(N_state):
        pop_traj[i, 0, :] = np.real(np.vdot(psi_initial, projectors[i] @ psi_initial))
        coh_traj[i, 0, :] = np.vdot(psi_initial, projectors_cohe[i] @ psi_initial)

    for traj in prange(N_traj):
        np.random.seed(seeds[traj])
        psi = psi_initial.copy()

        for step in range(1, n_times):
            # 1. Deterministic evolution given by the isolated System Hamiltonian
            psi = U_site @ psi

            # 2. Apply Kraus operator M0 to test the probability
            v0 = M0 @ psi
            
            # The probability P0 is exactly the squared norm of the resulting vector
            P0 = np.real(np.vdot(v0, v0))
            
            # 3. Stochastic jump
            r = np.random.rand()
            if r < P0:
                psi = v0 # M0 was already applied to v0
            else:
                psi = M1 @ psi

            # 4. State Normalization
            psi = psi / np.linalg.norm(psi)

            # 5. Extract observable values
            for i in range(N_state):
                pop_traj[i, step, traj] = np.real(np.vdot(psi, projectors[i] @ psi))
                coh_traj[i, step, traj] = np.vdot(psi, projectors_cohe[i] @ psi)

    return pop_traj, coh_traj

def compute_trajectory_wf(c_CM, dt, N_traj, times, 
                          projectors, projectors_cohe, psi_sys_initial, U_site, 
                          mode="QJ", batch_size=1000):
    """
    Wrapper function to handle batching and random seeds before calling the JIT core.
    """
    U_site = U_site.full() if hasattr(U_site, 'full') else np.array(U_site, dtype=complex)
    psi_sys_initial = psi_sys_initial.full() if hasattr(psi_sys_initial, 'full') else np.array(psi_sys_initial, dtype=complex)
    if psi_sys_initial.ndim > 1:
        psi_sys_initial = psi_sys_initial.flatten()
        
    projectors = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors])
    projectors_cohe = np.array([P.full() if hasattr(P, 'full') else np.array(P, dtype=complex) for P in projectors_cohe])

    n_times = len(times)
    
    # Generate the specific Kraus Operators
    M0, M1 = generate_kraus_operators(c_CM, dt, mode)

    rng_seeds = np.random.RandomState(42)
    all_seeds = rng_seeds.randint(0, 2**30, size=N_traj)

    pop_00 = np.zeros((n_times, N_traj), dtype=np.float64)
    pop_11 = np.zeros((n_times, N_traj), dtype=np.float64)
    coh_01 = np.zeros((n_times, N_traj), dtype=np.complex128)
    coh_10 = np.zeros((n_times, N_traj), dtype=np.complex128)

    N_done = 0
    n_batches = int(np.ceil(N_traj / batch_size))

    for b in range(n_batches):
        N_batch = min(batch_size, N_traj - N_done)
        seeds_b = all_seeds[N_done : N_done + N_batch]

        pop_b, coh_b = compute_trajectory_wf_core_single(
            psi_sys_initial, U_site, M0, M1, projectors, projectors_cohe,
            N_batch, n_times, seeds_b)

        pop_00[:, N_done : N_done + N_batch] = pop_b[0, :, :]
        pop_11[:, N_done : N_done + N_batch] = pop_b[1, :, :]
        
        coh_01[:, N_done : N_done + N_batch] = coh_b[0, :, :]
        coh_10[:, N_done : N_done + N_batch] = coh_b[1, :, :]

        N_done += N_batch
        del pop_b, coh_b

    return pop_00, pop_11, coh_01, coh_10

## Main Loop for varying $ dt $ and $ N_{traj} $

In [ ]:
# ===================
# System's Parameters
# ===================
np.random.seed(1) # always use the same seed 
N_site = 1    # Number of sites
#V_array = [1.0]    NO potential
Delta_E = 1.5 + np.random.randn(N_site)*0.1     #random inizialization of the system energies

# =========================
# Time Evolution Parameters
# =========================
dt_list = [0.01]     # change : time step
tf = 100.0    # Final Time
steps_list = [ int(tf / dt_list[i]) for i in range (len(dt_list)) ]
times_list = [ np.linspace(0, tf, int(steps_list[i])) for i in range(len(dt_list))]

N_traj_list = [20000]

# ===================
# Dephasing Parameter 
# ===================
g_decay = 0.1   # Gamma rate for the decay
# Lindblad Rates list
gamma_k = [g_decay]

# Scaling for the collsional algorithm c = sqrt(gamma / 4dt)
c_CM_list = np.array([np.sqrt(g_decay / (4 * dt_list[j])) for j in range(len(dt_list))] )  


In [ ]:
# ========================================
# Initial wave function and density matrix
# ========================================

# ======
# System
# ======
psi_sys_initial = basis(2, 0)

rho_sys_initial = (ket2dm(psi_sys_initial)).full()       

# =======
# Ancilla
# =======
psi_anc_single = basis(2,0) 
rho_anc_single = (ket2dm(psi_anc_single)).full()

# ==========
# Projectors
# ==========
P0 = (np.eye(2, dtype=complex) + sz) / 2 # Projector on |0>
P1 = (np.eye(2, dtype=complex) - sz) / 2 # Projector on |1>

# Tracking populations and coherences for the single qubit
projectors = np.array([P0, P1], dtype=complex) 
projectors_cohe = np.array([sm, sp], dtype=complex) 

# ======================
# Lindblad Jump Operator
# ======================
# Jump operator sigma minus
L_k = [sm]

# =================
# Angles definition
# =================

MODE = "close_to_90_deg"  # change : "normal" or "close_to_90_deg"

if MODE == "normal":
    theta_list = np.radians([90, 60, 45, 30, 0])
elif MODE == "close_to_90_deg": 
    theta_list = np.radians([90, 89.9, 89.5, 89, 88.5, 88])

### Calculation

In [ ]:
# ======================
# Output directory setup
# ======================

if MODE == "normal":
    results_dir = "../../Results/Data/Complete_rho/normal/"
elif MODE == "close_to_90_deg":
    results_dir = "../../Results/Data/Complete_rho/close_90_deg/"

os.makedirs(results_dir, exist_ok=True)

# Traiettorie per batch nel wrapper
BATCH_SIZE = 1000

def _make_fname_npz(results_dir, theta, dt, N_traj):
    t_str  = f"{theta:.6f}".replace(".", "p")
    dt_str = f"{dt:.6f}".replace(".", "p")
    return os.path.join(results_dir, f"result_theta{t_str}_dt{dt_str}_Ntraj{N_traj}.npz")

def _already_done_npz(results_dir, theta, dt, N_traj):
    return os.path.isfile(_make_fname_npz(results_dir, theta, dt, N_traj))

# =====================
# Main computation loop
# =====================
print("Starting computation for different theta, dt and N_traj")

for theta_idx, theta in enumerate(theta_list):

    print("=" * 50)
    print(f"THETA = {theta:.4f} rad = {np.degrees(theta):.2f}° ({theta_idx+1}/{len(theta_list)})")
    print("=" * 50)

    phi = theta - np.pi/2
    g_z = np.cos(theta)
    g_x = np.sin(theta)
    g_0 = np.cos(phi / 2)
    g_1 = np.sin(phi / 2)

    print(f"phi = {phi:.4f} rad, g_z = {g_z:.4f}, g_x = {g_x:.4f}, g_0 = {g_0:.4f}, g_1 = {g_1:.4f}")

    #Initialization of the Ancilla state for the collisional model (single ancilla, then tensorized for N_site)
    psi_anc_single = (g_0 * basis(2,0) + g_1 * basis(2,1))
    rho_anc_single = ket2dm(psi_anc_single)

    # Initialization of the probabilities for the trajectory WF evolution (same for all trajectories, depends only on theta and dt)
    Pr_0_list = np.zeros((N_site, len(dt_list)))
    Pr_1_list = np.zeros((N_site, len(dt_list)))

    for j in range(len(dt_list)):
        for i in range(N_site):
            Pr_0_list[i,j] = (g_0**2) * (np.cos(c_CM_list[i,j]*dt_list[j]))**2 + ((g_0 * g_z + g_1 * g_x)**2) * (np.sin(c_CM_list[i,j]*dt_list[j]))**2
            Pr_1_list[i,j] = (g_1**2) * (np.cos(c_CM_list[i,j]*dt_list[j]))**2 + ((g_0 * g_x - g_1 * g_z)**2) * (np.sin(c_CM_list[i,j]*dt_list[j]))**2

    for dt_idx, dt in enumerate(dt_list):

        print("=" * 40)
        print(f"dt = {dt:.4f} ({dt_idx+1}/{len(dt_list)})")

        times     = times_list[dt_idx]
        steps     = steps_list[dt_idx]
        c_CM      = c_CM_list[:, dt_idx]
        Pr_0_site = Pr_0_list[:, dt_idx]
        Pr_1_site = Pr_1_list[:, dt_idx]

        print("Recalculating Hamiltonians")
        H_site, H_coll, H_tot = hamiltonian_N_ancillas(N_site, E, V_array, c_CM, g_x, g_z)
        U_tot, U_diag, w, V = evolution_operator(H_tot, dt, method='diagonalization', hermitian=True)
        U_diag_dag = U_diag.conj().T; V_dag = V.conj().T
        H_system = system_Hamiltonian(N_site, E, V_array, mode="complete")
        U_site, U_diag_site, w_site, V_site = evolution_operator(H_system, dt, method='diagonalization', hermitian=True)

        print("Computing Lindblad")
        start_time = time.time()
        rho_list_lindblad, V_lindblad, W_lindblad = Lindblad_evo(rho_sys_initial, H_system, gamma_k, L_k, times, method="diagonal", vectorized=False)
        print(f"Completed in {time.time() - start_time:.2f}s")

        print("Computing Trajectory Isolated")
        start_time = time.time()
        pop_traj_isolated = compute_trajectory_wf_isolated(N_site, times, projectors, psi_sys_initial, U_site)
        print(f"Completed in {time.time() - start_time:.2f}s")

        print("Computing Trace Ancilla")
        start_time = time.time()
        pops_trace = compute_trace_ancilla(rho_sys_initial, rho_anc_single, U_diag, V, times, projectors, N_site)
        print(f"Completed in {time.time() - start_time:.2f}s")

        for N_traj in N_traj_list:

            print("-" * 40)
            print(f"N_traj = {N_traj}")

            if _already_done_npz(results_dir, theta, dt, N_traj):
                print("  Already done, skipping.")
                continue

            print("Computing Trajectory WF (All Distributions)")
            start_time = time.time()

            pop_00, pop_11, coh_01_10, coh_10_01 = compute_trajectory_wf(
                dt, c_CM, g_z, g_x, g_0, g_1, N_traj, N_site, times,
                projectors, projectors_cohe, psi_sys_initial, U_site, Pr_0_site, Pr_1_site,
                batch_size=BATCH_SIZE)

            print(f"Completed in {time.time() - start_time:.2f}s")

            fname_npz = _make_fname_npz(results_dir, theta, dt, N_traj)

            np.savez_compressed(
                fname_npz,
                # 1. Dati Raw Traiettorie (Distribuzioni)
                pop_00=pop_00,
                pop_11=pop_11,
                coh_01_10=coh_01_10, 
                coh_10_01=coh_10_01,
                
                # 2. Baseline Analitiche e Traccia
                pops_trace=pops_trace,
                rho_list_lindblad=rho_list_lindblad,
                V_lindblad=V_lindblad,
                W_lindblad=W_lindblad,
                
                # 3. Dati Sistema Isolato
                pop_traj_isolated=pop_traj_isolated,
                
                # 4. Parametri
                theta=theta, phi=phi, dt=dt, N_traj=N_traj,
                times=times, steps=steps, c_CM=c_CM,
                g_z=g_z, g_x=g_x, g_0=g_0, g_1=g_1)

            size_mb = os.path.getsize(fname_npz) / (1024**2)
            print(f"  Saved → {os.path.basename(fname_npz)}  ({size_mb:.2f} MB)")

            # Pulizia immediata della RAM 
            del pop_00, pop_11, coh_01_10, coh_10_01

        del rho_list_lindblad, pop_traj_isolated

print("\n" + "=" * 40)
print("COMPUTATION COMPLETED!")
print(f"Results saved for:")
print(f"  - {len(theta_list)} theta values: {[f'{t:.4f}' for t in theta_list]}")
print(f"  - {len(dt_list)} dt values: {dt_list}")
print(f"  - {len(N_traj_list)} N_traj values: {N_traj_list}")
print("=" * 40)

Starting computation for different theta, dt and N_traj
THETA = 1.5708 rad = 90.00° (1/5)
phi = 0.0000 rad, g_z = 0.0000, g_x = 1.0000, g_0 = 1.0000, g_1 = 0.0000
dt = 0.0100 (1/1)
Recalculating Hamiltonians
Computing Lindblad
Completed in 0.27s
Computing Trajectory Isolated
Completed in 0.02s
Computing Trace Ancilla
Completed in 0.11s
----------------------------------------
N_traj = 20000
Computing Trajectory WF (All Distributions)
Completed in 57.15s
  Saved → result_theta1p570796_dt0p010000_Ntraj20000.npz  (6811.92 MB)
THETA = 1.0472 rad = 60.00° (2/5)
phi = -0.5236 rad, g_z = 0.5000, g_x = 0.8660, g_0 = 0.9659, g_1 = -0.2588
dt = 0.0100 (1/1)
Recalculating Hamiltonians
Computing Lindblad
Completed in 0.02s
Computing Trajectory Isolated
Completed in 0.01s
Computing Trace Ancilla
Completed in 0.11s
----------------------------------------
N_traj = 20000
Computing Trajectory WF (All Distributions)
Completed in 50.36s
  Saved → result_theta1p047198_dt0p010000_Ntraj20000.npz  (8693.73 